In [11]:
import json
from pathlib import Path
import numpy as np
import requests
import pandas as pd

In [13]:
league_id = 'kfszz7vdmdl0krou'
season = '15'
latest_gw = 39

In [25]:
session = requests.Session()

# 1. Hit the base league page first to establish valid session cookies/tokens
initial_url = f"https://www.fantrax.com/fantasy/league/{league_id}/standings"
session.get(initial_url, headers={"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"})

records = []

for period in range(1, latest_gw):
    url = f'https://www.fantrax.com/fxpa/req?leagueId={league_id}'

    headers = {
        "accept": "application/json",
        "content-type": "text/plain",
        "origin": "https://www.fantrax.com",
        "referer": f"https://www.fantrax.com/fantasy/league/{league_id}/standings;view=SCHEDULE;timeframeType=BY_PERIOD;timeStartType=FROM_SEASON_START;period={period}",
        "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
    }
    
    payload = {
        "msgs": [
            {
                "method": "getStandings",
                "data": {
                    "leagueId": league_id,
                    "view": "REGULAR_SEASON",
                    "timeframeType": "BY_PERIOD",
                    "timeStartType": "FROM_SEASON_START",
                    "period": str(period)
                }
            }
        ],
        "uiv": 3,
        "refUrl": f"https://www.fantrax.com/fantasy/league/{league_id}/standings;view=SCHEDULE;timeframeType=BY_PERIOD;timeStartType=FROM_SEASON_START;period={period}",
        "dt": 1,
        "at": 0,
        "av": "0.0",
        "tz": "America/Los_Angeles",
        "v": "185.0.1"  # Ensure this matches the current active web build version if hardcoded
    }

    r = session.post(url, headers=headers, json=payload)
    r.raise_for_status()
    j = r.json()

    data = j["responses"][0]["data"]
    team_info = data.get("fantasyTeamInfo", {})

    # find the table for the current week table
    standings_tbl = next(
        (t for t in data.get("tableList", []) if t.get("caption") == f"Gameweek {period}"),
        None
    )
    if not standings_tbl:
        print(f"Period {period}: Head-to-head table not found")
        continue
    
    for row in standings_tbl['rows']:
        away_team   = row["cells"][0]["content"]
        away_score  = row["cells"][1]["content"]
        home_team   = row["cells"][2]["content"]
        home_score  = row["cells"][3]["content"]

        records.append({
            "gw": period,
            "away_team": away_team,
            "home_team": home_team,
            "away_score": float(away_score),
            "home_score": float(home_score),
        })
    
df = pd.DataFrame(records)

{'data': {'sDate': 1785618715078}, 'roles': ['03'], 'responses': [{'data': {'goBackDays': [1, 7, 14, 30, 60], 'fantasyTeamInfo': {'frxhyucbmdl0krp3': {'name': 'Seanhampton', 'logoUrl512': 'https://fantraximg.com/logos/d0i/tmLogo_d0iynm50me9ewuv9_512.webp', 'shortName': 'SHFC'}, 'nax64rt3mdl0krp3': {'name': 'FPL 5: AutoPick Strikes Back', 'logoUrl512': 'https://fantraximg.com/assets/images/icons/fantasyteams/soccer/shoe/blue_teal_256.webp', 'shortName': 'APSB'}, 'xngo7h0mmdl0krp3': {'name': 'Thottenham Hotsluts', 'logoUrl512': 'https://fantraximg.com/logos/v95/tmLogo_v95dat5xm003tufu_512.jpg', 'shortName': 'THHS'}, 'q0m6hwmimdl0krp3': {'name': 'GAK-PO-TAY-TOES', 'logoUrl512': 'https://fantraximg.com/logos/mnf/tmLogo_mnf675temk7t6ugi_512.webp', 'shortName': 'TJ'}, 'bdh1g8b8mdl0krp3': {'name': 'Walton Goggonzola', 'logoUrl512': 'https://fantraximg.com/assets/images/icons/fantasyteams/soccer/gloves/lightblue_pink_512.webp', 'shortName': 'Zac'}, '71z5x96fmdl0krp3': {'name': 'Benford FC', 'l

In [26]:
# list of all teams
teams = sorted(df["home_team"].unique())

# Step 1 — Determine winner/loser
def get_result(row):
    if row["away_score"] > row["home_score"]:
        return row["away_team"], row["home_team"]
    elif row["home_score"] > row["away_score"]:
        return row["home_team"], row["away_team"]
    else:
        return None, None  # tie

df["winner"], df["loser"] = zip(*df.apply(get_result, axis=1))

# Step 2 — Build win records
wins = df.dropna(subset=["winner"]).loc[:, ["winner", "loser"]]
wins["count"] = 1

# Step 3 — Aggregate counts
win_counts = (
    wins.groupby(["winner", "loser"])["count"]
        .sum()
        .reset_index()
)

# Step 4 — Ensure all teams appear as losers for every winner
all_pairs = pd.MultiIndex.from_product([teams, teams], names=["winner", "loser"]).to_frame(index=False)
win_counts = all_pairs.merge(win_counts, on=["winner", "loser"], how="left").fillna(0).rename(
    columns={'count':'wins'}
)
win_counts["wins"] = win_counts["wins"].astype(int)

In [27]:
# Create mapping from team to index
team_index = {team: i for i, team in enumerate(teams)}

# Add x/y coordinates
win_counts["x"] = win_counts["loser"].map(team_index)
win_counts["y"] = win_counts["winner"].map(team_index)

In [28]:
win_counts.loc[win_counts["winner"] == win_counts["loser"], "wins"] = ""

In [29]:
win_counts.to_csv(f'data/output/h2h/season-{season}.csv', index=False)

In [30]:
chart_data = win_counts.to_csv(index=False)

In [31]:
name_to_short = {v["name"]: v["shortName"] for v in team_info.values()}

In [32]:
name_to_short["Mattchester United"] = "MU"

In [33]:
access_token = 'yXNpSzS8XN1dCvPgQUS2ofNz3IvncMH26TkRLZe2stW0fCDxXZAnCufjAOfRNiDc'
chart_id = '4Qpyc'
base_url = 'https://api.datawrapper.de/v3/charts'
chart_url = f'{base_url}/{chart_id}'
chart_data_url = f'{chart_url}/data'
publish_url = f'{chart_url}/publish'

json_headers={
    'accept': '*/*',
    'Authorization':f'Bearer {access_token}',
    'content-type':'application/json' 
}
csv_headers={
    'accept': '*/*',
    'Authorization':f'Bearer {access_token}',
    'content-type':'text/csv'
}

Send latest data to Datawrapper

In [34]:
update_data_response = requests.put(chart_data_url, data=chart_data, headers=csv_headers)

Generate x-axis annotations

In [35]:
x_axis_annotions = []
for name, index in team_index.items():
    short_name = name_to_short[name]    
    anno_obj = {
        'dx': 0,
        'dy': 0,
        'id': f'x-{short_name}-{index}',
        'size': 10,
        'text': short_name,
        'align': 'bc',
        'width': 12.5,
        'position': {'x': index, 'y': '-0.5'},
    }
    x_axis_annotions.append(anno_obj)

Generate y-axis annotations

In [36]:
y_axis_annotions = []
for name, index in team_index.items():
    short_name = name_to_short[name]    
    anno_obj = {
        'dx': 0,
        'dy': 0,
        'id': f'y-{short_name}-{index}',
        'size': 10,
        'text': short_name,
        'align': 'mr',
        'width': 12.5,
        'position': {'x': '-0.55', 'y': index},
    }
    y_axis_annotions.append(anno_obj)

Set color scale responsive to max no. of wins

In [37]:
max_wins = max_wins = win_counts['wins'].where(win_counts['wins'] != '', 0).astype(int).max()
max_wins

4

In [38]:
WIN_COLORSETS = {
    0: ['#ffffcc'],
    1: ['#ffffcc', '#c2e699'],
    2: ['#ffffcc', '#c2e699', '#78c679'],
    3: ['#ffffcc', '#c2e699', '#78c679', '#238443'],
    4: ['#ffffcc', '#c2e699', '#78c679', '#238443', '#006837'],
    5: ['#ffffcc', '#d9f0a3', '#addd8e', '#78c679', '#31a354', '#006837'],
}

DEFAULT_COLORSET = ['#f7fcb9', '#addd8e', '#31a354']

def make_color_scheme(max_wins):
    colors = WIN_COLORSETS.get(max_wins, DEFAULT_COLORSET)

    color_map = {str(i): colors[i] for i in range(len(colors))}
    color_map[""] = "#ffffff"

    return {
        "map": color_map,
        "categoryLabels": {
            str(i): f"{i} wins" if i == max_wins and max_wins > 1 else str(i)
            for i in range(len(colors))
        }
    }

color_categories = make_color_scheme(max_wins)

Send to Datawrapper

In [39]:
payload = {
    "metadata": {
        "visualize": {
            "color-category": color_categories,
            "text-annotations": x_axis_annotions + y_axis_annotions
        }
    }
}

In [40]:
response = requests.patch(chart_url, json=payload, headers=json_headers)
response

<Response [200]>

In [41]:
publish_chart_response = requests.post(publish_url,headers=json_headers)
publish_chart_response

<Response [200]>

In [42]:
latest_chart_version = publish_chart_response.json()['data']['publicVersion']
latest_chart_version

18

In [43]:
filepath = Path('../_data/charts.json')
with filepath.open("r", encoding="utf-8") as f:
    charts = json.load(f)

charts[season]['h2h']['version'] = latest_chart_version

with filepath.open("w", encoding="utf-8") as f:
    json.dump(charts, f, indent=2)

In [44]:
form_melted = df[df.gw >= (latest_gw-5)].melt(id_vars=['gw', 'winner', 'loser'], 
                 value_vars=['away_team', 'home_team'], 
                 value_name='team')

# 2. Define the form logic
def get_form(row):
    if row['winner'] == row['team']:
        return '🟩'
    elif row['loser'] == row['team']:
        return '🟥'
    else:
        return '⬜'

form_melted['form'] = form_melted.apply(get_form, axis=1)

In [45]:
form_table = form_melted.pivot(index='team', columns='gw', values='form')
form_table.index.name = 'Team'

In [46]:
print(form_table.to_csv())

Team,34,35,36
Benford FC,🟩,🟩,🟩
Cheasle FC,🟥,🟥,🟩
FPL 5: AutoPick Strikes Back,🟩,🟥,🟥
GAK-PO-TAY-TOES,🟩,🟩,🟥
Magpies United,🟥,🟩,🟩
Mattchester United,🟥,🟩,🟩
Morning Timber,🟩,🟥,🟥
Seanhampton,🟩,🟩,🟩
Thottenham Hotsluts,🟥,🟥,🟥
Walton Goggonzola,🟥,🟥,🟥

